# Data Acquisition, Merge, and Checks

Given the large dataset, several notebooks were used to originally concatenate the data from NYISO, pull data from ERA5, format and standardize the data, and conduct initial data checks. The code below is a simplified version of the code across notebooks from Google Colab, Kaggle, and Jupyter Notebook. I experimented with different tools to determine which would best handle this large dataset with the least amount of lag during the initial stage of data acquisition, merging, and checking. That is why this code is mostly in markdown.

## Imports and Configuration

In [18]:
import os
import zipfile
from pathlib import Path

import pandas as pd

## Paths

In [19]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
raw_dir = data_root / "raw"
processed_dir = data_root / "processed"

solar_raw_dir = raw_dir / "nyiso_solar"

In [20]:
nyiso_out = processed_dir / "01_nyiso_merged.csv"
era5_src = processed_dir / "02_era5_weather.csv"
era5_out = processed_dir / "02_era5_weather.csv"
merged_out = processed_dir / "03_merged_data.csv"

processed_dir.mkdir(parents=True, exist_ok=True)

In [27]:
solar_zip_path = raw_dir / "nyiso_solar.zip"

unzipped_dirs = {
    "actuals": solar_raw_dir / "unzipped_actuals",
    "forecasts": solar_raw_dir / "unzipped_forecasts",
    "capacity": solar_raw_dir / "unzipped_capacity",
}

In [28]:
if solar_zip_path.exists():
    solar_raw_dir.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(solar_zip_path, "r") as archive:
            archive.extractall(solar_raw_dir)
        print(f"Extracted Main: {solar_zip_path}")
    except Exception as e:
        print(f"Didn't Extract Main: {solar_zip_path.name} | {e}")
else:
    print(f"Not Found: {solar_zip_path}")

Extracted Main: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar.zip


In [30]:
def unzip_all_archives(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    extracted_files = 0

    if not os.path.exists(input_folder):
        print(f"Input folder not found: {input_folder}")
        return

    for filename in os.listdir(input_folder):
        if filename.endswith(".zip"):
            try:
                with zipfile.ZipFile(os.path.join(input_folder, filename), "r") as archive:
                    archive.extractall(output_folder)
                    extracted_files += 1
            except Exception as e:
                print(f"Did Not Extract: {filename} | {e}")

    print(f"Extraction Completed: {extracted_files} archives from {input_folder}")

for key in nyiso_dirs:
    unzip_all_archives(nyiso_dirs[key], unzipped_dirs[key])

def load_folder(folder: Path) -> pd.DataFrame:
    csv_files = list(folder.glob("*.csv"))
    frames = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df["source_file"] = file.name
            frames.append(df)
        except Exception as e:
            print(f"Failed to Read: {file.name} | {e}")

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames, ignore_index=True)
    return df_all

Input folder not found: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar/NYISO_Actuals
Input folder not found: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar/NYISO_Forecasts
Input folder not found: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar/NYISO_Capacity


In [24]:
df_actual = load_folder(unzipped_dirs["actuals"])
df_forecast = load_folder(unzipped_dirs["forecasts"])
df_capacity = load_folder(unzipped_dirs["capacity"])

In [25]:
print("Loaded Shapes")
print(". . .")
print("Actuals:", df_actual.shape)
print("Forecasts:", df_forecast.shape)
print("Capacity:", df_capacity.shape)

Loaded Shapes
. . .
Actuals: (0, 0)
Forecasts: (0, 0)
Capacity: (0, 0)


In [26]:
for df in (df_actual, df_forecast, df_capacity):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

AttributeError: Can only use .str accessor with string values, not integer

In [4]:
ts_col = "time_stamp"
zone_col = "zone_name"
val_col = "mw_value"

In [ ]:
for df in (df_actual, df_forecast, df_capacity):
    df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")

    if pd.api.types.is_datetime64tz_dtype(df[ts_col]):
        df[ts_col] = df[ts_col].dt.tz_convert("UTC")
    else:
        df[ts_col] = df[ts_col].dt.tz_localize(
            "America/New_York",
            nonexistent="shift_forward",
            ambiguous="NaT",
        ).dt.tz_convert("UTC")

    df[ts_col] = df[ts_col].dt.floor("H")

for df in (df_actual, df_forecast, df_capacity):
    df[zone_col] = df[zone_col].astype(str).str.upper().str.strip()

for df in (df_actual, df_forecast, df_capacity):
    df[val_col] = pd.to_numeric(df[val_col], errors="coerce")

In [ ]:
df_actual = (
    df_actual
    .dropna(subset=[ts_col, zone_col, val_col])
    .groupby([ts_col, zone_col], as_index=False)[val_col]
    .sum()
    .rename(columns={val_col: "actual_mw"})
)

df_forecast = (
    df_forecast
    .dropna(subset=[ts_col, zone_col, val_col])
    .groupby([ts_col, zone_col], as_index=False)[val_col]
    .sum()
    .rename(columns={val_col: "forecast_mw"})
)

df_capacity = (
    df_capacity
    .dropna(subset=[ts_col, zone_col, val_col])
    .groupby([ts_col, zone_col], as_index=False)[val_col]
    .sum()
    .rename(columns={val_col: "capacity_mw"})

In [ ]:
df_nyiso = (
    df_actual
    .merge(df_forecast, how="outer", on=[ts_col, zone_col])
    .merge(df_capacity, how="outer", on=[ts_col, zone_col])
    .sort_values([ts_col, zone_col])
    .reset_index(drop=True)
)

df_nyiso.to_csv(nyiso_out, index=False)

In [ ]:
df_era5 = pd.read_csv(era5_src, low_memory=False)

df_era5.columns = (
    df_era5.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

if ts_col not in df_era5.columns:
    if "time" in df_era5.columns:
        df_era5[ts_col] = pd.to_datetime(df_era5["time"], utc=True, errors="coerce")
    else:
        raise ValueError("ERA5 must contain 'time' or 'time_stamp' column")
else:
    df_era5[ts_col] = pd.to_datetime(df_era5[ts_col], utc=True, errors="coerce")

df_era5[ts_col] = df_era5[ts_col].dt.floor("H")
df_era5 = df_era5.dropna(subset=[ts_col])

df_era5 = (
    df_era5
    .sort_values(ts_col)
    .groupby(ts_col, as_index=False)
    .first()
)

df_era5.to_csv(era5_out, index=False)

In [ ]:
df_merge = pd.merge(df_nyiso, df_era5, on=ts_col, how="inner")
df_merge = df_merge.sort_values([ts_col, zone_col]).reset_index(drop=True)

df_merge.to_csv(merged_out, index=False)

print("Merged Shape:", df_merge.shape)
print("Merged Time Range:", df_merge[ts_col].min(), "→", df_merge[ts_col].max())
print("Saved:", merged_out)

In [ ]:
print("Columns Check")
print(". . .")
print("NYISO Columns:", df_nyiso.columns.tolist())
print("ERA5 Columns:", df_era5.columns.tolist())
print("Merged Columns:", df_merge.columns.tolist())

In [ ]:
print("Missing Check")
print(". . .")
print("NYISO Missing time_stamp:", df_nyiso[ts_col].isna().sum())
print("ERA5 Missing time_stamp:", df_era5[ts_col].isna().sum())
print("Merged Missing time_stamp:", df_merge[ts_col].isna().sum())

In [ ]:
print("Duplication Check")
print(". . .")
print("NYISO Duplicates:", df_nyiso.duplicated(subset=[ts_col, zone_col]).sum())
print("ERA5 Duplicates:", df_era5.duplicated(subset=[ts_col]).sum())
print("Merged Duplicates:", df_merge.duplicated(subset=[ts_col, zone_col]